# Création d'un sous-dataset stratifié

Création d'un sous-dataset stratifié (~15%) à partir :
- d'un ZIP contenant les audios WAV
- d'un ZIP contenant les spectrogrammes associés

**Sortie:**
- `3s_samples_subset.zip`
- `3s_mel_spectrograms_subset.zip`

In [1]:
import os
import re
import zipfile

from collections import defaultdict, Counter
from pathlib import Path

from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("Imports successful")

Imports successful


## Configuration

In [2]:
AUDIO_ZIP = r"C:\Users\Simon\Desktop\CNRS\Data\Audio\AudioRaw\3s_samples.zip"

SPEC_ZIP = r"C:\Users\Simon\Desktop\CNRS\Data\Audio\Spectrogramme\3s_mel_spectrograms.zip"

OUTPUT_AUDIO_DIR = r"C:\Users\Simon\Desktop\CNRS\Data\Audio\AudioRaw\3s_samples_subset2"

OUTPUT_SPEC_DIR = r"C:\Users\Simon\Desktop\CNRS\Data\Audio\Spectrogramme\3s_mel_spectrograms_subset2"

SAMPLE_RATIO = 0.15

RANDOM_STATE = 42

MIN_SAMPLES_PER_CLASS = 10

print("Configuration loaded")

Configuration loaded


## Fonction pour extraire le label

In [3]:
def extract_label(filename):
    """
    Extrait le label avant '_HiP'
    """

    match = re.match(r"^(.*?)_HiP", filename)

    if match:
        return match.group(1)

    return "unknown"


# Test
test_filename = "example_label_HiP123_idx456.wav"
print(f"Test: {test_filename} -> {extract_label(test_filename)}")

Test: example_label_HiP123_idx456.wav -> example_label


## Lecture des fichiers ZIP

In [4]:
print("Lecture des fichiers ZIP")

with zipfile.ZipFile(AUDIO_ZIP, 'r') as audio_zip:

    audio_files = [
        f for f in audio_zip.namelist()
        if f.endswith(".wav")
    ]

print(f"Nombre de fichiers audio trouvés : {len(audio_files)}")

Lecture des fichiers ZIP
Nombre de fichiers audio trouvés : 68970


## Création des labels

In [5]:
labels = []
filenames = []

for file in audio_files:

    filename = os.path.basename(file)

    label = extract_label(filename)

    filenames.append(filename)
    labels.append(label)

print(f"Labels créés pour {len(filenames)} fichiers")

Labels créés pour 68970 fichiers


## Analyse distribution des labels

In [6]:
label_counts = Counter(labels)

print("\nDistribution initiale des labels :\n")

for label, count in sorted(label_counts.items()):

    print(f"{label:<30} {count}")

print(f"\nNombre total de labels : {len(label_counts)}")


Distribution initiale des labels :

alarm_call                     211
alarm_call_o                   5
baboon                         240
background                     15952
bird                           1190
bird_flight                    1
breath                         3263
breath_hi                      2027
breath_hi_o                    1
buffalo                        26
burp                           50
cough                          1
crunch                         12180
cub_vocalise                   1
cub_whining                    40
dog                            187
donkey                         9
drink                          356
drink_o                        2
firearm                        4
flies                          514
flowing_water                  7
frog                           29
giggle                         53
giggle_o                       129
gretting_groal                 86
gretting_groal_o               5
groan                          398
gr

## Identification des labels rares

In [7]:
rare_labels = [
    label
    for label, count in label_counts.items()
    if count < MIN_SAMPLES_PER_CLASS
]

if rare_labels:

    print("\nLabels rares supprimés :\n")

    for label in rare_labels:
        print(f"{label:<30} {label_counts[label]}")

else:
    print("\nAucun label rare détecté.")


Labels rares supprimés :

alarm_call_o                   5
bird_flight                    1
breath_hi_o                    1
cough                          1
cub_vocalise                   1
donkey                         9
drink_o                        2
firearm                        4
flowing_water                  7
gretting_groal_o               5
growl                          2
growl_thr                      6
growl_thr_o                    6
hiccup                         7
jap_o                          5
lion_roar_period               8
prey_run                       1
rain                           2
roar_resp_period               1
rumble_o                       5
run_water                      1
scream                         1
scream_prey                    7
sneeze                         2
snort                          4
trot_water                     2
underwater                     4
urinate                        5
walk_o                         1
walk_water_o    

## Filtrage des labels rares

In [8]:
filtered_filenames = []
filtered_labels = []

for file, label in zip(filenames, labels):

    if label_counts[label] >= MIN_SAMPLES_PER_CLASS:

        filtered_filenames.append(file)
        filtered_labels.append(label)

filenames = filtered_filenames
labels = filtered_labels

print(f"Nombre de fichiers après filtrage : {len(filenames)}")

Nombre de fichiers après filtrage : 68836


## Sampling stratifié

In [9]:
print("Sampling stratifié...")

selected_files, _, selected_labels, _ = train_test_split(
    filenames,
    labels,
    train_size=SAMPLE_RATIO,
    stratify=labels,
    random_state=RANDOM_STATE
)

selected_set = set(selected_files)

print(f"\nNombre total fichiers conservés : {len(filenames)}")
print(f"Nombre sélectionné : {len(selected_files)}")
print(f"Ratio : {len(selected_files) / len(filenames):.2%}")

Sampling stratifié...

Nombre total fichiers conservés : 68836
Nombre sélectionné : 10325
Ratio : 15.00%


## Création des dossiers de sortie

In [10]:
os.makedirs(OUTPUT_AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUT_SPEC_DIR, exist_ok=True)

print(f"Dossiers de sortie créés :")
print(f"  Audio: {OUTPUT_AUDIO_DIR}")
print(f"  Spec:  {OUTPUT_SPEC_DIR}")

Dossiers de sortie créés :
  Audio: C:\Users\Simon\Desktop\CNRS\Data\Audio\AudioRaw\3s_samples_subset2
  Spec:  C:\Users\Simon\Desktop\CNRS\Data\Audio\Spectrogramme\3s_mel_spectrograms_subset2


## Extraction des audios

In [11]:
print("Extraction des audios sélectionnés...")

with zipfile.ZipFile(AUDIO_ZIP, 'r') as audio_zip:

    for file in tqdm(audio_zip.namelist()):

        filename = os.path.basename(file)

        if filename in selected_set:

            output_path = os.path.join(
                OUTPUT_AUDIO_DIR,
                filename
            )

            with audio_zip.open(file) as source:
                with open(output_path, "wb") as target:
                    target.write(source.read())

print("Extraction des audios terminée.")

Extraction des audios sélectionnés...


100%|██████████| 68972/68972 [00:27<00:00, 2514.33it/s]

Extraction des audios terminée.


## Extraction des spectrogrammes

In [12]:
print("Extraction des spectrogrammes associés...")

with zipfile.ZipFile(SPEC_ZIP, 'r') as spec_zip:

    for file in tqdm(spec_zip.namelist()):

        filename = os.path.basename(file)

        # nom sans extension
        base_name = Path(filename).stem

        # nom audio correspondant
        wav_name = base_name + ".wav"

        if wav_name in selected_set:

            output_path = os.path.join(
                OUTPUT_SPEC_DIR,
                filename
            )

            with spec_zip.open(file) as source:
                with open(output_path, "wb") as target:
                    target.write(source.read())

print("Extraction des spectrogrammes terminée.")

Extraction des spectrogrammes associés...


100%|██████████| 68971/68971 [01:02<00:00, 1106.49it/s]

Extraction des spectrogrammes terminée.


## Statistiques finales

In [ ]:
print("\nDistribution finale des labels :\n")

distribution = defaultdict(int)

for label in selected_labels:
    distribution[label] += 1

for label, count in sorted(distribution.items()):

    print(f"{label:<30} {count}")

print("\nTerminé.")
print(f"Audios sauvegardés dans : {OUTPUT_AUDIO_DIR}")
print(f"Spectrogrammes sauvegardés dans : {OUTPUT_SPEC_DIR}")


Distribution finale des labels :

alarm_call                     32
baboon                         36
background                     2393
bird                           178
breath                         489
breath_hi                      304
buffalo                        4
burp                           7
crunch                         1827
cub_whining                    6
dog                            28
drink                          53
flies                          77
frog                           4
giggle                         8
giggle_o                       19
gretting_groal                 13
groan                          60
groan_o                        33
growl_res                      109
growl_res_o                    4
h_noise                        15
heartbeat                      78
human                          2
hyena                          20
insect                         30
l_noise                        52
lick                           234
lick_o     

## Vérification de la stratification - Proportions des labels

In [15]:
import pandas as pd

# Distribution initiale (après filtrage)
initial_distribution = Counter(labels)
initial_total = sum(initial_distribution.values())

# Distribution finale (sélectionnée)
final_distribution = Counter(selected_labels)
final_total = sum(final_distribution.values())

# Créer un tableau de comparaison
comparison_data = []

for label in sorted(set(labels)):
    initial_count = initial_distribution.get(label, 0)
    initial_percent = (initial_count / initial_total * 100) if initial_total > 0 else 0
    
    final_count = final_distribution.get(label, 0)
    final_percent = (final_count / final_total * 100) if final_total > 0 else 0
    
    diff_percent = final_percent - initial_percent
    
    comparison_data.append({
        'Label': label,
        'Initial (count)': initial_count,
        'Initial (%)': initial_percent,
        'Final (count)': final_count,
        'Final (%)': final_percent,
        'Diff (%)': diff_percent
    })

# Afficher sous forme de tableau
comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("VÉRIFICATION DE LA STRATIFICATION - PROPORTIONS DES LABELS")
print("="*100)
print(f"\nBase de données initiale : {initial_total} échantillons")
print(f"Base de données créée   : {final_total} échantillons")
print(f"Ratio de conservation   : {final_total / initial_total * 100:.2f}%\n")

print(comparison_df.to_string(index=False))

# Résumé de la préservation des proportions
print("\n" + "="*100)
print("RÉSUMÉ")
print("="*100)
max_diff = comparison_df['Diff (%)'].abs().max()
mean_diff = comparison_df['Diff (%)'].abs().mean()

print(f"Écart maximal de proportion     : {max_diff:.4f}%")
print(f"Écart moyen de proportion       : {mean_diff:.4f}%")
print(f"\nStratification : {'RÉUSSIE' if mean_diff < 1 else 'À VÉRIFIER'} (écart moyen < 1%)")


VÉRIFICATION DE LA STRATIFICATION - PROPORTIONS DES LABELS

Base de données initiale : 68836 échantillons
Base de données créée   : 10325 échantillons
Ratio de conservation   : 15.00%

                Label  Initial (count)  Initial (%)  Final (count)  Final (%)  Diff (%)
           alarm_call              211     0.306526             32   0.309927  0.003402
               baboon              240     0.348655             36   0.348668  0.000014
           background            15952    23.173921           2393  23.176755  0.002835
                 bird             1190     1.728747            178   1.723971 -0.004776
               breath             3263     4.740252            489   4.736077 -0.004175
            breath_hi             2027     2.944680            304   2.944310 -0.000370
              buffalo               26     0.037771              4   0.038741  0.000970
                 burp               50     0.072636              7   0.067797 -0.004840
               crunch 